# Prepare the CCLE / DepMap multi-omics archive

This reference notebook performs the one-time, data-heavy preparation step.
It keeps each assay on its native cell-line axis in a `MuData` and then wraps
that object in Celldega's `DatasetCollection`.  An unmeasured assay is never
represented as an all-zero row.

Raw files are expected in `data/ccle_multiomics/raw/`, which is ignored by
git.  The required inputs are the DepMap 24Q2 `Model.csv`, batch-corrected
protein-coding RNA expression, Chronos gene effects, and the CCLE RPPA and
metabolomics CSV files. The CCLE MS proteomics file is detected and added as
a separate modality when present. See `ccle_multiomics/README.md` for
filenames and sources.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys


PROJECT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from notebooks.ccle_multiomics import build_collection, load_collection


RAW = PROJECT / "data/ccle_multiomics/raw"
ARCHIVE = PROJECT / "data/ccle_multiomics/processed/ccle_multiomics.h5mu"

/Users/nicolasfernandez/Documents/celldega/dega/lib/python3.12/site-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version


/Users/nicolasfernandez/Documents/celldega/dega/lib/python3.12/site-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/Users/nicolasfernandez/Documents/celldega/dega/lib/python3.12/site-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


Set `REBUILD = True` only when raw inputs change.  The resulting H5MU stores
native assay coverage: RNA has more models than the legacy RPPA panel, for
example, rather than being reduced immediately to a complete-case cohort.

In [2]:
REBUILD = False

if REBUILD or not ARCHIVE.exists():
    collection = build_collection(
        RAW,
        ARCHIVE,
        include_crispr=True,
        include_proteomics_ms=None,
        overwrite=REBUILD,
    )
else:
    collection = load_collection(ARCHIVE)

print(type(collection).__name__)
for modality, adata in collection.mod.items():
    print(f"{modality:14s} {adata.n_obs:4d} models x {adata.n_vars:6d} features")

DatasetCollection
rna            1517 models x  19137 features
metabolomics    927 models x    225 features
rppa            894 models x    214 features
crispr         1150 models x  18443 features
